# 01. 학습 기록(access_log) 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Reasonofmoon/hangeorum-playground/blob/main/notebooks/01_access_log_analysis.ipynb)

**이 노트북으로 할 일**
- 시트의 `access_log` 탭을 읽습니다.
- 날짜별 조회 수, 많이 본 튜토리얼, 사용자 수, 할당량 초과(`result` = `429`) 횟수를 봅니다.

**먼저 알아 둘 것**
- 읽기만 합니다. 시트에 아무것도 쓰지 않습니다.
- 이메일은 개수를 셀 때만 씁니다. 화면에 이메일 목록을 출력하지 않습니다.
- 그래프 글자 깨짐을 막으려고 축 이름은 영어로 씁니다.

**순서**: 위에서부터 셀을 하나씩 실행하세요. (단축키: Shift + Enter)

In [ ]:
# 1) 내 Google 계정으로 인증합니다. 창이 뜨면 시트를 볼 수 있는 계정을 고르고 허용하세요.
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
print("인증 완료")

In [ ]:
#@title 2) 시트 ID 입력
#@markdown 시트 주소 `https://docs.google.com/spreadsheets/d/<여기>/edit` 의 `<여기>` 부분만 넣습니다.
SHEET_ID = "" #@param {type:"string"}
assert SHEET_ID.strip(), "SHEET_ID를 입력하세요."
sh = gc.open_by_key(SHEET_ID.strip())
print("열린 시트:", sh.title)

## 3) `access_log` 읽기
열: `ts, email, trace, action, paths, tutorial_id, result, ms`

In [ ]:
import pandas as pd

ws = sh.worksheet("access_log")
df = pd.DataFrame(ws.get_all_records())
print("행 개수:", len(df))

expected_cols = ["ts", "email", "trace", "action", "paths", "tutorial_id", "result", "ms"]
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    print("빠진 열:", missing)
    for c in missing:
        df[c] = ""

# 형식 맞추기
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df["result"] = df["result"].astype(str).str.strip()
df["ms"] = pd.to_numeric(df["ms"], errors="coerce")
df["email"] = df["email"].astype(str).str.strip().replace("", pd.NA)
df["tutorial_id"] = df["tutorial_id"].astype(str).str.strip()
df["date"] = df["ts"].dt.date
print("날짜를 읽지 못한 행:", int(df["ts"].isna().sum()))

# 미리보기 (이메일 열은 숨깁니다)
df.drop(columns=["email"]).head()

## 4) 요약 숫자

In [ ]:
summary = pd.DataFrame({
    "항목": ["전체 기록", "사용자 수(고유 이메일)", "튜토리얼 수", "기간 시작", "기간 끝", "429 횟수"],
    "값": [
        len(df),
        int(df["email"].dropna().nunique()),
        int(df.loc[df["tutorial_id"] != "", "tutorial_id"].nunique()),
        str(df["ts"].min()),
        str(df["ts"].max()),
        int((df["result"] == "429").sum()),
    ],
})
summary

## 5) 날짜별 조회 수와 사용자 수

In [ ]:
import matplotlib.pyplot as plt

valid = df.dropna(subset=["ts"])
daily = valid.groupby("date").size()
users_daily = valid.groupby("date")["email"].nunique()

fig, ax = plt.subplots(figsize=(10, 4))
daily.plot(ax=ax, marker="o", label="views")
users_daily.plot(ax=ax, marker=".", label="unique users")
ax.set_title("Daily views and users")
ax.set_xlabel("Date")
ax.set_ylabel("Count")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6) 많이 본 튜토리얼 TOP 10
막대 이름은 `tutorial_id`(영문 ID)라서 글자가 깨지지 않습니다.

In [ ]:
top = (df[df["tutorial_id"] != ""]
       .groupby("tutorial_id")
       .agg(views=("tutorial_id", "size"), users=("email", "nunique"))
       .sort_values("views", ascending=False)
       .head(10))

if top.empty:
    print("tutorial_id가 있는 기록이 없습니다.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    top["views"].iloc[::-1].plot.barh(ax=ax)
    ax.set_title("Top 10 tutorials by views")
    ax.set_xlabel("Views")
    ax.set_ylabel("tutorial_id")
    plt.tight_layout()
    plt.show()
top

## 7) 할당량 초과(429) 확인
`result`가 `429`인 기록은 공개 API의 호출 한도에 걸린 요청입니다. 많으면 캐시나 한도 설정을 점검하세요.

In [ ]:
q = df[df["result"] == "429"]
print("429 횟수:", len(q))
if len(q):
    q_daily = q.dropna(subset=["ts"]).groupby("date").size()
    fig, ax = plt.subplots(figsize=(10, 3))
    q_daily.plot.bar(ax=ax, color="tab:red")
    ax.set_title("Quota hits (result == 429) per day")
    ax.set_xlabel("Date")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()
    display(q.groupby("action").size().rename("count").to_frame())
else:
    print("429 기록이 없습니다.")

## 8) 결과 상태별 개수와 응답 시간

In [ ]:
display(df["result"].value_counts().rename("count").to_frame())
print("응답 시간(ms) 요약")
print(df["ms"].describe()[["count", "mean", "50%", "max"]])

---
**다음 단계**
- 같은 데이터를 대시보드로 계속 보려면 GW04(Looker Studio, 현재 이름 Data Studio)를 따라 하세요.
- 막히면: 인증 셀(1번)부터 다시 실행합니다. `SpreadsheetNotFound`는 ID나 권한 문제, `WorksheetNotFound`는 탭 이름 문제입니다.